
# CS650 — Assignment #2: **Redis Advanced Data Modeling and Performance**

### 📘 Overview
This assignment includes two major components:  
1. **🧪 Guided Lab** — A hands-on exercise that introduces Redis data structures, indexing, transactions, and real-time data streams step-by-step.  
2. **📝 Assignment Tasks** — A set of challenges to extend your learning independently after completing the lab.

The lab focuses on progressively building your understanding of Redis modeling concepts — starting from simple key-value structures to more advanced event-driven pipelines — while the assignment section evaluates your ability to reason about design trade-offs, concurrency, and performance.


## 📑 How to Use This Notebook
- Run cells **top → bottom**. The lab uses a local Redis connection (e.g., `redis-py`). If you don't have Redis running, comment the connection and skim outputs.
- Lab cells are meant to be **executed** as-is; do not modify unless prompted.
- Complete **Part 2** in the provided code cells under each task.


---

## 🎯 Learning Objectives
By the end of this assignment, you should be able to:
- Apply Redis data structures (Strings, Hashes, Sets, Sorted Sets, Streams) for domain modeling.
- Build secondary indexes to support fast queries.
- Use transactions (`MULTI/EXEC`) and optimistic locking (`WATCH`) to manage concurrent updates.
- Implement simple Pub/Sub or Streams for event-driven pipelines.
- Measure Redis performance and reason about trade-offs in data modeling.

---

## 🧪 Guided Lab
Follow the guided lab steps below to explore Redis commands and modeling techniques. Each section introduces one new concept and builds on the previous one.


## 🧪 Lab 0 — Environment & Redis Setup

To keep this assignment portable, we begin by ensuring a running Redis service and a Python client.  
- **If you're on your laptop** and already have Redis running on `localhost:6379`, you can skip the first cell and just run the connection cell.  
- **If you're in Google Colab**, run the first two cells to install/start Redis and the Python client, then run the connection cell.

This setup creates a reusable client `r` used throughout the guided lab and assignment tasks.

In [ ]:
# Lab 0 — Start Redis (Colab-only). If Redis runs locally, you may skip this.
!apt-get -qq install redis-server > /dev/null
!redis-server --daemonize yes
!redis-cli ping

PONG


In [ ]:
# Lab 0 — Install Python Redis client (redis-py)
!pip install -q redis

In [ ]:
# Lab 0 — Connect to Redis and verify
import redis

r = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)

try:
    print("Redis connection:", "OK" if r.ping() else "FAILED")
except Exception as e:
    print("❌ Connection failed. Ensure Redis server is running (Colab users: run the cells above).")
    print("Error:", e)

Redis connection: OK



## 🔧 Namespace Setup

To keep Redis keys isolated for this lab, we define a reusable namespace prefix `NS`.  
All Redis keys will use this prefix (e.g., `a2:user:1`, `a2:order:9001`) to prevent accidental conflicts with other datasets.


In [ ]:

# Define namespace prefix for all Redis keys in this assignment
NS = "a2"
print(f"Using Redis namespace prefix: {NS}")


Using Redis namespace prefix: a2


# ✅ Part 1 — Guided Lab (Execute Only)

### 1) Keys & Hashes (Entities)

**Step Continuation**  
This step advances the lab workflow, extending the previous state so results remain connected and easy to verify.

In [ ]:
# Patients and doctors as hashes
r.hset(f"{NS}:patient:1", mapping={'name':'Alex Chen','dob':'1990-03-01'})
r.hset(f"{NS}:patient:2", mapping={'name':'Priya Rao','dob':'1987-09-12'})
r.hset(f"{NS}:doctor:1", mapping={'name':'Dr. Carter','specialty':'Cardiology'})
r.hset(f"{NS}:doctor:2", mapping={'name':'Dr. Singh','specialty':'Pediatrics'})
print(r.hgetall(f"{NS}:patient:1"))
print(r.hgetall(f"{NS}:doctor:1"))


{'name': 'Alex Chen', 'dob': '1990-03-01'}
{'name': 'Dr. Carter', 'specialty': 'Cardiology'}


### 2) Sets for Secondary Indexes

**Step Continuation**  
This step advances the lab workflow, extending the previous state so results remain connected and easy to verify.

In [ ]:
# Secondary indexes: patients by birth year, doctors by specialty
r.sadd(f"{NS}:idx:patient:dob:1990", f"{NS}:patient:1")
r.sadd(f"{NS}:idx:patient:dob:1987", f"{NS}:patient:2")
r.sadd(f"{NS}:idx:doctor:specialty:Cardiology", f"{NS}:doctor:1")
r.sadd(f"{NS}:idx:doctor:specialty:Pediatrics", f"{NS}:doctor:2")
print(r.smembers(f"{NS}:idx:doctor:specialty:Cardiology"))


{'a2:doctor:1'}


### 3) Sorted Sets for Scheduling & Leaderboards

**Step Continuation**  
This step advances the lab workflow, extending the previous state so results remain connected and easy to verify.

In [ ]:
# Appointments in a sorted set (score = epoch seconds)
from datetime import datetime
def to_epoch(dt_str):
    return int(datetime.fromisoformat(dt_str).timestamp())

appt_key = f"{NS}:zset:appointments"
r.zadd(appt_key, {f"{NS}:appt:1": to_epoch("2025-10-25T09:00:00")})
r.zadd(appt_key, {f"{NS}:appt:2": to_epoch("2025-10-25T10:00:00")})
print(r.zrange(appt_key, 0, -1, withscores=True))


[('a2:appt:1', 1761382800.0), ('a2:appt:2', 1761386400.0)]


### 4) TTLs & Expiration

**Step Continuation**  
This step advances the lab workflow, extending the previous state so results remain connected and easy to verify.

In [ ]:
# Temporary hold for an appointment slot with TTL (5 minutes)
hold_key = f"{NS}:hold:doc:1:2025-10-25T09:00"
r.set(hold_key, f"{NS}:patient:1", ex=300)
print("TTL:", r.ttl(hold_key))


TTL: 300


### 5) Pipelines & Transactions (WATCH/MULTI/EXEC)

**Transactions with Optimistic Locking**  
A small purchasing routine demonstrates `WATCH`/`MULTI`/`EXEC`. The operation retries if the watched value changes between read and commit.

In [ ]:
# Optimistic booking with WATCH/MULTI/EXEC
slot_key = f"{NS}:slot:doc:1:2025-10-25T09:00"
while True:
    try:
        r.watch(slot_key)
        if r.exists(slot_key):
            r.unwatch()
            print("Already booked")
            break
        p = r.pipeline()
        p.multi()
        p.set(slot_key, f"{NS}:patient:1")
        p.execute()
        print("Booked")
        break
    except redis.WatchError:
        continue


Already booked


### 6) Streams for Audit Logging

**Streams as an Append‑Only Log**  
Redis Streams record an ordered event trail. We append a few order lifecycle events and read them back to verify the flow.

In [ ]:
# Audit event via Streams
stream = f"{NS}:stream:clinic:audit"
r.xadd(stream, {'evt':'create_appt','doctor':'1','patient':'1'})
print(r.xrange(stream, '-', '+')[-1])


('1788811224169-0', {'evt': 'create_appt', 'doctor': '1', 'patient': '1'})


# 🧪 Part 2 — Assignment Tasks (Redis, Clinic Domain)

> **Instructions:** Complete the three tasks below in the **clinic** domain using Redis.  
> Keep implementations concise; prefer idiomatic Redis patterns discussed in Part 1.

## Task 1 — Entities & Indexes

Design Redis keys for **patients**, **doctors**, and **appointments** using hashes.  
Create **secondary indexes** so you can look up:  
- Doctors by `specialty`  
- Appointments by `doctor` and by **day** (hint: sets or sorted sets with date prefix)

_Note_: Briefly justify your key naming conventions and index choice (2–3 sentences).


**Step Continuation**  
This step advances the lab workflow, extending the previous state so results remain connected and easy to verify.

In [ ]:
# Task 1 — Create patients, doctors, appointments, and secondary indexes

# Patients
r.hset(f"{NS}:patient:101", mapping={
    "name": "Jordan Lee",
    "dob": "1994-06-15"
})

r.hset(f"{NS}:patient:102", mapping={
    "name": "Maria Gomez",
    "dob": "1989-11-03"
})

r.hset(f"{NS}:patient:103", mapping={
    "name": "Kevin Brown",
    "dob": "2001-02-20"
})

# Doctors
r.hset(f"{NS}:doctor:101", mapping={
    "name": "Dr. Wilson",
    "specialty": "Dermatology"
})

r.hset(f"{NS}:doctor:102", mapping={
    "name": "Dr. Patel",
    "specialty": "Neurology"
})

# Appointments
appointments = {
    "101": {"patient_id": "101", "doctor_id": "101", "date": "2026-09-10", "time": "09:00"},
    "102": {"patient_id": "102", "doctor_id": "101", "date": "2026-09-10", "time": "10:00"},
    "103": {"patient_id": "103", "doctor_id": "102", "date": "2026-09-10", "time": "11:00"},
    "104": {"patient_id": "101", "doctor_id": "102", "date": "2026-09-11", "time": "13:00"}
}

for appt_id, data in appointments.items():
    appt_key = f"{NS}:appt:{appt_id}"
    r.hset(appt_key, mapping=data)

    # Index appointments by doctor
    r.sadd(f"{NS}:idx:appt:doctor:{data['doctor_id']}", appt_key)

    # Index appointments by day
    date_key = data["date"].replace("-", "")
    r.sadd(f"{NS}:idx:appt:date:{date_key}", appt_key)

# Index doctors by specialty
r.sadd(f"{NS}:idx:doctor:specialty:Dermatology", f"{NS}:doctor:101")
r.sadd(f"{NS}:idx:doctor:specialty:Neurology", f"{NS}:doctor:102")

# Proof
print("Dermatology doctors:",
      r.smembers(f"{NS}:idx:doctor:specialty:Dermatology"))

print("Appointments for doctor 101:",
      r.smembers(f"{NS}:idx:appt:doctor:101"))

print("Appointments on 2026-09-10:",
      r.smembers(f"{NS}:idx:appt:date:20260910"))

Dermatology doctors: {'a2:doctor:101'}
Appointments for doctor 101: {'a2:appt:101', 'a2:appt:102'}
Appointments on 2026-09-10: {'a2:appt:103', 'a2:appt:101', 'a2:appt:102'}


I used hashes for patients, doctors, and appointments because each record has several related fields. I used a consistent key format with the a2 namespace so the data is easy to identify. Sets are used as secondary indexes because they make it easy to find doctors by specialty and appointments by doctor or date.

## Task 2 — Booking Operation (Atomic)

Implement a **booking operation** for a doctor/time slot that prevents double-booking.  
Use either **WATCH/MULTI/EXEC** or a small **Lua** script to ensure atomicity.

_Note_: Write 2–3 lines explaining how your solution prevents race conditions.


**Transactions with Optimistic Locking**  
A small purchasing routine demonstrates `WATCH`/`MULTI`/`EXEC`. The operation retries if the watched value changes between read and commit.

In [ ]:
# Task 2 — Atomic booking using WATCH/MULTI/EXEC

slot_key = f"{NS}:slot:doc:101:2026-09-12T09:00"
patient_key = f"{NS}:patient:102"

while True:
    try:
        with r.pipeline() as pipe:
            # Watch the slot for changes made by another client
            pipe.watch(slot_key)

            if pipe.exists(slot_key):
                print("Booking failed: slot is already booked.")
                pipe.unwatch()
                break

            pipe.multi()
            pipe.set(slot_key, patient_key)
            pipe.execute()

            print("Booking successful.")
            break

    except redis.WatchError:
        print("Slot changed during booking. Retrying...")
        continue

print("Current booking:", r.get(slot_key))

Booking successful.
Current booking: a2:patient:102


WATCH monitors the appointment slot before the transaction is completed. If another user changes the slot before EXEC, Redis cancels the transaction and the code retries. This prevents two patients from booking the same doctor and time slot at the same time.

## Task 3 — Time-bound Holds & Activity Log

Add a **temporary hold** for a slot with a **TTL** (e.g., 5 minutes).  
When a booking is confirmed, **append an event** to a Redis **Stream** `stream:clinic:audit`.

_Note_: Provide a one-line `XRANGE` to show the latest event you added.


**Step Continuation**  
This step advances the lab workflow, extending the previous state so results remain connected and easy to verify.

In [ ]:
# Task 3 — Temporary hold with TTL and audit stream

hold_key = f"{NS}:hold:doc:102:2026-09-12T14:00"
patient_key = f"{NS}:patient:103"

# Hold the slot for 5 minutes
r.set(hold_key, patient_key, ex=300)

print("Hold created:", r.get(hold_key))
print("Hold TTL:", r.ttl(hold_key))

# Confirm the booking
confirmed_slot = f"{NS}:slot:doc:102:2026-09-12T14:00"
r.set(confirmed_slot, patient_key)

# Remove temporary hold after confirmation
r.delete(hold_key)

# Add confirmation event to audit stream
stream = f"{NS}:stream:clinic:audit"

event_id = r.xadd(stream, {
    "event": "booking_confirmed",
    "doctor_id": "102",
    "patient_id": "103",
    "slot": "2026-09-12T14:00"
})

print("Audit event added:", event_id)

# Show the latest audit event
print("Latest audit event:", r.xrange(stream, "-", "+")[-1])

Hold created: a2:patient:103
Hold TTL: 300
Audit event added: 1788811224222-0
Latest audit event: ('1788811224222-0', {'event': 'booking_confirmed', 'doctor_id': '102', 'patient_id': '103', 'slot': '2026-09-12T14:00'})


The temporary hold uses a TTL of 300 seconds, so it automatically expires if the patient does not complete the booking. After the booking is confirmed, the hold is removed and an event is added to the Redis Stream to keep an audit record of the activity.

---
## ✅ Final Checklist
- [ ] Task 1: Entities modeled and indexes created with a short justification.
- [ ] Task 2: Atomic booking implemented (WATCH/MULTI/EXEC or Lua) with a short note.
- [ ] Task 3: TTL-based hold and stream event recorded; quick proof via a read.
- [ ] Notebook runs top→bottom; outputs visible.

## 📄 Submission
- Export a **PDF** of this notebook **and** submit the `.ipynb` file on Canvas.
- Ensure **all outputs** are visible in the PDF.

## 🧮 Grading Rubric (100 points)
| Criteria | Points | Description |
|---|---:|---|
| **Task 1 — Modeling & Indexes** | 35 | Clean keys, correct hashes, useful secondary indexes. |
| **Task 2 — Atomic Booking** | 35 | Correct use of transactions or Lua to avoid double-booking. |
| **Task 3 — TTL + Streams** | 20 | Effective hold semantics and audit event written. |
| **Clarity & Organization** | 10 | Brief justifications, clean code, reproducible. |